In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [3]:
df1 = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
df2 = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
df = df2.merge(df1,on=['TransactionID'],how='left')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), object(31)
memory usage: 1.9+ GB


In [4]:
from sklearn.model_selection import GroupKFold

X = df.drop(columns=['isFraud'])
y = df['isFraud']

groups = X['card2'].astype(str) + "-" + X['addr2'].astype(str) + "-" + X['C1'].astype(str)

gkf = GroupKFold(n_splits=5)

train_idx, test_idx = next(gkf.split(X, y, groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

# Cleaning

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class DealWithNans(BaseEstimator, TransformerMixin):
    def __init__(self, drop_threshold=0.8):
        self.drop_threshold = drop_threshold
        self.fill_values_ = {}
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        self.cols_to_drop_ = [
            col for col in X.columns if X[col].isna().mean() >= self.drop_threshold
        ]

        X_filtered = X.drop(columns=self.cols_to_drop_)

        for col in X_filtered.columns:
            if X_filtered[col].dtype == "object" or str(X_filtered[col].dtype).startswith("category"):
                # Categorical: use mode
                mode = X_filtered[col].mode()
                self.fill_values_[col] = mode.iloc[0] if not mode.empty else np.nan
            else:
                # Numeric: use median
                self.fill_values_[col] = X_filtered[col].median()

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X = X.drop(columns=self.cols_to_drop_, errors="ignore")
        return X.fillna(self.fill_values_)


In [7]:
cleaner = DealWithNans(drop_threshold = 0.8)
cleaner.fit(X_train, y_train)
X_no_nans = cleaner.transform(X_train)

In [23]:
!pip install mlflow dagshub
import mlflow
import dagshub

In [24]:
dagshub.init(repo_owner='mr-master-afk', repo_name='ML-Fraud-detection', mlflow=True)

Accessing as mr-master-afk

Initialized MLflow to track repo "mr-master-afk/ML-Fraud-detection"

Repository mr-master-afk/ML-Fraud-detection initialized!

In [10]:
experiment_name = "XGBoost"
run_name = "XGBoost_cleaning"
# Set the experiment name
mlflow.set_experiment(experiment_name)
removed_col_cnt = X_train.shape[1] - X_no_nans.shape[1]
with mlflow.start_run(run_name=run_name):
    mlflow.log_param("drop_threshold", 0.8)
    mlflow.log_param("dropped_columns_cnt", removed_col_cnt)
    mlflow.sklearn.log_model(
        cleaner,
        artifact_path="cleaner_model",
        registered_model_name="Cleaner",
    )
mlflow.end_run()

2025/04/29 16:08:23 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost' does not exist. Creating a new experiment.
2025/04/29 16:08:24 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/04/29 16:08:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Cleaner' already exists. Creating a new version of this model...
2025/04/29 16:08:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Cleaner, version 2
Created version '2' of model 'Cleaner'.


🏃 View run XGBoost_cleaning at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1/runs/f33ebaa5ff0b4a629f8c7ea0700eeaf5
🧪 View experiment at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1


# Feature engineering


In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

class CustomFeatureEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.label_encoders = {}

    def fit(self, X, y=None):
        for feature in X.select_dtypes(include=['object', 'category']).columns:
            encoder = LabelEncoder()
            encoder.fit(X[feature])
            self.label_encoders[feature] = encoder
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for feature, encoder in self.label_encoders.items():
            unseen_labels = ~X_transformed[feature].isin(encoder.classes_)
            if unseen_labels.any():
                X_transformed.loc[unseen_labels, feature] = encoder.classes_[0]
            X_transformed[feature] = encoder.transform(X_transformed[feature])

        return X_transformed


In [13]:
scaler_and_encoder = Pipeline([
    ('encoder', CustomFeatureEncoder()),
    ('scaler', StandardScaler())
])
scaler_and_encoder.fit(X_no_nans, y_train)
X_enc_sc = scaler_and_encoder.transform(X_no_nans)
with mlflow.start_run(run_name="Encoder_and_Scaler") as run:
    mlflow.sklearn.log_model(
        scaler_and_encoder,
        artifact_path="encoder_and_scaler_model",
        registered_model_name="Encoder_and_Scaler",
    )
mlflow.end_run()

2025/04/29 16:11:54 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/04/29 16:11:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Encoder_and_Scaler' already exists. Creating a new version of this model...
2025/04/29 16:12:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Encoder_and_Scaler, version 2
Created version '2' of model 'Encoder_and_Scaler'.


🏃 View run Encoder_and_Scaler at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1/runs/21717304e4a94caa92d7dcde93c0017c
🧪 View experiment at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1


# Feature Selection

In [10]:
class CorrelationFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, cnt):
        self.cnt = cnt
        self.selected_features = []

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = pd.Series(y)

        corrs = X.apply(lambda col: np.abs(np.corrcoef(col, y)[0, 1]))
        
        self.selected_features = corrs.nlargest(self.cnt).index.tolist()
        return self

    def transform(self, X):
        return pd.DataFrame(X)[self.selected_features]

In [19]:
corr_feature_selector = CorrelationFeatureSelector(200)
corr_feature_selector.fit(X_enc_sc, y_train)
X_corr_selected = corr_feature_selector.transform(X_enc_sc)
with mlflow.start_run(run_name="Feature_Selection_Correlation") as run:
    mlflow.sklearn.log_model(
        corr_feature_selector,
        artifact_path="feature_selection_corr_xgboost",
        registered_model_name="Feature_Selection_Corr_Xgboost",
    )
    mlflow.log_param("count_of_features", corr_feature_selector.cnt)
mlflow.end_run()


2025/04/29 16:18:07 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/04/29 16:18:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Feature_Selection_Corr_Xgboost' already exists. Creating a new version of this model...
2025/04/29 16:18:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Feature_Selection_Corr_Xgboost, version 2
Created version '2' of model 'Feature_Selection_Corr_Xgboost'.


🏃 View run Feature_Selection_Correlation at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1/runs/f72a8223fa414220bcc7f27a01cbe4b9
🧪 View experiment at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1


# Training

In [34]:
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from collections import Counter

counter = Counter(y_train)
neg, pos = counter[0], counter[1]
scale_pos_weight = neg / pos
pipeline = Pipeline(steps=[
    ('imputer', DealWithNans(drop_threshold = 0.95)),
    ('encoder', CustomFeatureEncoder()),
    ('feature_selector', CorrelationFeatureSelector(400)),
    ('classifier', XGBClassifier(
        n_estimators=500,         
        max_depth=5,              
        learning_rate=0.03,        
        subsample=0.8,            
        colsample_bytree=0.6,     
        scale_pos_weight=scale_pos_weight,       
        use_label_encoder=False,  
        eval_metric='auc',        
        random_state=42,          
        n_jobs=-1
    ))   
])
pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', DealWithNans(drop_threshold=0.95)),
                ('encoder', CustomFeatureEncoder()),
                ('feature_selector', CorrelationFeatureSelector(cnt=400)),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.6, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.03,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=500, n_jobs=-1,
                               num_parallel_tree=None, random_state=42, ...))])

In [35]:
from sklearn.metrics import roc_auc_score

y_prob_train = pipeline.predict_proba(X_train)[:, 1]
y_prob_test = pipeline.predict_proba(X_test)[:, 1]
roc_auc_train = roc_auc_score(y_train, y_prob_train)
roc_auc_test = roc_auc_score(y_test, y_prob_test)
roc_auc_train, roc_auc_test

(0.9399750524692567, 0.886324864072785)

In [36]:
experiment_name = "XGBoost"
mlflow.set_experiment(experiment_name)
import mlflow
with mlflow.start_run(run_name="XGBoost_model") as run:
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="XGBoost_pipeline_model",
        registered_model_name="XGBoost_pipeline_model"
    )
    mlflow.log_metric("roc_test", roc_auc_test)
    mlflow.log_metric("roc_train", roc_auc_train)
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.03)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.6)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_param("use_label_encoder", False)
    mlflow.log_param("eval_metric", "auc")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_jobs", -1)

2025/04/29 17:34:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'XGBoost_pipeline_model' already exists. Creating a new version of this model...
2025/04/29 17:34:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_pipeline_model, version 4
Created version '4' of model 'XGBoost_pipeline_model'.


🏃 View run XGBoost_model at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1/runs/b57b85143ada415199699a70f3be1ef8
🧪 View experiment at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/1
